In [ ]:
import json
import os
import pandas as pd
import torch
import re
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel
import numpy as np
from sklearn.model_selection import train_test_split
import gc
from typing import Dict, List, Tuple, Optional

class MedicalKnowledgeEnhancer:
    """Enhances model with differential diagnosis generation"""
    
    def __init__(self, symptoms_to_diseases_csv: str = None):
        """Initialize with optional symptoms-to-diseases mapping"""
        self.symptoms_to_diseases = {}
        if symptoms_to_diseases_csv and os.path.exists(symptoms_to_diseases_csv):
            self._load_symptoms_mapping(symptoms_to_diseases_csv)
    
    def _load_symptoms_mapping(self, csv_path: str):
        """Load symptoms to diseases mapping"""
        df = pd.read_csv(csv_path)
        # Assuming columns: symptom, disease, frequency or similar
        for _, row in df.iterrows():
            symptom = str(row['symptom']).lower().strip()
            disease = str(row['disease']).lower().strip()
            if symptom not in self.symptoms_to_diseases:
                self.symptoms_to_diseases[symptom] = []
            self.symptoms_to_diseases[symptom].append(disease)
    
    def extract_symptoms_from_note(self, clinical_note: str) -> List[str]:
        """Extract key symptoms from clinical note"""
        # Common medical symptoms (simplified - you can expand this)
        common_symptoms = [
            'fever', 'cough', 'headache', 'fatigue', 'pain', 'nausea', 
            'vomiting', 'diarrhea', 'shortness of breath', 'chest pain',
            'dizziness', 'rash', 'swelling', 'bleeding', 'infection',
            'hypertension', 'tachycardia', 'bradycardia', 'hypotension',
            'edema', 'cyanosis', 'jaundice', 'pallor'
        ]
        
        symptoms_found = []
        clinical_note_lower = clinical_note.lower()
        
        for symptom in common_symptoms:
            if symptom in clinical_note_lower:
                symptoms_found.append(symptom)
        
        return symptoms_found[:5]  # Return top 5 symptoms
    
    def generate_differential_diagnoses(self, primary_disease: str, symptoms: List[str]) -> List[str]:
        """Generate potential differential diagnoses"""
        differentials = []
        
        # Based on symptoms, suggest related conditions
        symptom_based_differentials = []
        for symptom in symptoms:
            if symptom in self.symptoms_to_diseases:
                symptom_based_differentials.extend(self.symptoms_to_diseases[symptom])
        
        # Common medical differentials (simplified mapping)
        common_differentials = {
            'pneumonia': ['bronchitis', 'influenza', 'covid-19', 'tuberculosis', 'lung cancer'],
            'myocardial infarction': ['angina', 'aortic dissection', 'pulmonary embolism', 'pericarditis'],
            'stroke': ['migraine', 'seizure', 'hypoglycemia', 'brain tumor'],
            'diabetes': ['hyperthyroidism', 'cushing syndrome', 'pheochromocytoma'],
            'appendicitis': ['gastroenteritis', 'uti', 'ovarian cyst', 'diverticulitis'],
            'asthma': ['copd', 'bronchitis', 'heart failure', 'anxiety'],
            'migraine': ['tension headache', 'cluster headache', 'sinusitis', 'meningitis'],
            'depression': ['bipolar disorder', 'anxiety disorder', 'thyroid disorder', 'dementia'],
            'hypertension': ['renal artery stenosis', 'cushing syndrome', 'pheochromocytoma'],
            'arthritis': ['gout', 'lupus', 'fibromyalgia', 'osteoporosis']
        }
        
        # Add symptom-based differentials
        if symptom_based_differentials:
            differentials.extend(list(set(symptom_based_differentials))[:3])
        
        # Add common differentials for the primary disease
        primary_lower = primary_disease.lower()
        for disease_pattern, diffs in common_differentials.items():
            if disease_pattern in primary_lower or primary_lower in disease_pattern:
                differentials.extend(diffs)
        
        # Remove duplicates and the primary disease itself
        differentials = [d for d in set(differentials) if d.lower() != primary_disease.lower()]
        
        return differentials[:5]  # Return top 5 differentials

class PrecautionsEvaluator:
    """Handles loading and evaluating precautions from CSV"""
    
    def __init__(self, precautions_csv_path: str):
        self.precautions_df = pd.read_csv(precautions_csv_path)
        self.precautions_dict = self._create_precautions_dict()
    
    def _create_precautions_dict(self) -> Dict[str, str]:
        precautions_dict = {}
        for _, row in self.precautions_df.iterrows():
            disease = str(row['disease']).lower().strip()
            if 'precautions' in row:
                precautions = str(row['precautions'])
            else:
                precaution_cols = [col for col in row.index if 'precaution' in col.lower()]
                if precaution_cols:
                    precautions = ' | '.join([str(row[col]) for col in precaution_cols if pd.notna(row[col])])
                else:
                    precautions = ""
            precautions_dict[disease] = precautions
        return precautions_dict
    
    def get_precautions(self, disease: str) -> str:
        return self.precautions_dict.get(disease.lower().strip(), "")
    
    def evaluate_precautions_match(self, generated_text: str, true_disease: str) -> Dict[str, float]:
        true_precautions = self.get_precautions(true_disease)
        
        if not true_precautions or not generated_text:
            return {"match_score": 0.0, "has_precautions": 0.0}
        
        generated_precautions = self._extract_precautions_from_text(generated_text)
        
        if not generated_precautions:
            return {"match_score": 0.0, "has_precautions": 0.0}
        
        true_keywords = set(true_precautions.lower().split())
        gen_keywords = set(generated_precautions.lower().split())
        
        if not true_keywords:
            return {"match_score": 0.0, "has_precautions": 1.0}
        
        intersection = len(true_keywords.intersection(gen_keywords))
        union = len(true_keywords.union(gen_keywords))
        match_score = intersection / union if union > 0 else 0.0
        
        return {
            "match_score": match_score,
            "has_precautions": 1.0,
            "true_precautions": true_precautions[:200],
            "generated_precautions": generated_precautions[:200]
        }
    
    def _extract_precautions_from_text(self, text: str) -> str:
        patterns = [
            r"Precautions:\s*(.+)",
            r"Precautions for [^:]+:\s*(.+)",
            r"\*\*Precautions:\*\*\s*(.+)",
            r"Precautions[\s\-:]+(.+)"
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
            if match:
                precautions = match.group(1).strip()
                for end_marker in ["\n\n**", "\n\n[", "\n\nNote:", "\n\nDiagnostic"]:
                    if end_marker in precautions:
                        precautions = precautions.split(end_marker)[0]
                return precautions
        
        return ""

def load_model_with_frozen_adapters(tokenizer_path: str = "./qwen1.5b-symptoms-precautions"):
    """Load model with frozen phase 1 adapters"""
    
    print("Loading tokenizer from phase 1...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_path,
            trust_remote_code=True,
            use_fast=True
        )
        print("Successfully loaded tokenizer from phase 1")
    except:
        print("Using base model tokenizer")
        tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen2.5-1.5B-Instruct",
            trust_remote_code=True,
            use_fast=True
        )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    
    base_model.resize_token_embeddings(len(tokenizer))
    
    print("Loading and freezing phase 1 adapters...")
    try:
        phase1_adapters_path = os.path.join(tokenizer_path, "phase1_adapters")
        if os.path.exists(phase1_adapters_path):
            model = PeftModel.from_pretrained(
                base_model,
                phase1_adapters_path,
                is_trainable=False
            )
            print("Successfully loaded and froze phase 1 adapters")
        else:
            print("Phase 1 adapters not found, starting from base model")
            model = base_model
    except Exception as e:
        print(f"Error loading phase 1 adapters: {e}")
        model = base_model
    
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    
    lora_config_phase2 = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=4,
        lora_alpha=8,
        lora_dropout=0.05,
        target_modules=["gate_proj", "up_proj"],
        bias="none",
    )
    
    model = get_peft_model(model, lora_config_phase2)
    
    print("\nTrainable parameters (Phase 2 only):")
    model.print_trainable_parameters()
    
    return model, tokenizer

def load_training_data(csv_path: str, precautions_csv_path: str = None):
    """Load training data from CSV"""
    
    print(f"Loading training data from {csv_path}...")
    df = pd.read_csv(csv_path)
    
    required_cols = ['disease', 'clinical_note', 'reasoning_chain']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV must contain '{col}' column")
    
    print(f"Loaded {len(df)} examples")
    
    precautions_evaluator = None
    if precautions_csv_path and os.path.exists(precautions_csv_path):
        print(f"Loading precautions data from {precautions_csv_path}...")
        precautions_evaluator = PrecautionsEvaluator(precautions_csv_path)
    
    return df, precautions_evaluator

def preprocess_training_function_with_differential(
    examples, 
    tokenizer, 
    medical_knowledge: MedicalKnowledgeEnhancer = None,
    include_differential: bool = False
):
    """Preprocess training data with optional differential diagnosis"""
    
    texts = []
    
    for i in range(len(examples['clinical_note'])):
        clinical_note = examples['clinical_note'][i]
        reasoning_chain = examples['reasoning_chain'][i]
        disease = examples['disease'][i]
        
        clinical_note = clinical_note[:1500]
        
        if include_differential and medical_knowledge:
            # Extract symptoms for differential diagnosis
            symptoms = medical_knowledge.extract_symptoms_from_note(clinical_note)
            differentials = medical_knowledge.generate_differential_diagnoses(disease, symptoms)
            
            instruction = """Analyze this clinical note and provide:
1. Most suspected disease (primary diagnosis)
2. Differential diagnoses (other possible conditions to consider)
3. Diagnostic reasoning
4. Precautions for the primary diagnosis

Format your response as:
**Primary Diagnosis:** [disease name]
**Differential Diagnoses:** [list of other possible conditions]
**Reasoning:** [your reasoning chain]
**Precautions:** [precautions for primary diagnosis]"""
            
            if differentials:
                differential_text = ", ".join(differentials[:3])
            else:
                differential_text = "Consider other conditions with similar symptoms"
            
            target_response = f"""**Primary Diagnosis:** {disease}
**Differential Diagnoses:** {differential_text}
**Reasoning:** {reasoning_chain}
**Precautions:** [Precautions would be generated based on the primary diagnosis]"""
        
        else:
            # Standard training without differential
            instruction = """Analyze this clinical note and provide:
1. Most suspected disease
2. Diagnostic reasoning
3. Precautions for the most suspected disease

Format your response as:
**Disease:** [disease name]
**Reasoning:** [your reasoning chain]
**Precautions:** [precautions]"""
            
            target_response = f"""**Disease:** {disease}
**Reasoning:** {reasoning_chain}
**Precautions:** [Precautions would be generated based on the disease]"""
        
        full_text = f"<|im_start|>user\n{instruction}\n\nClinical Note:\n{clinical_note}<|im_end|>\n<|im_start|>assistant\n{target_response}<|im_end|>"
        texts.append(full_text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding=False,
        max_length=512,
        return_tensors=None,
        add_special_tokens=True,
    )
    
    result = {
        'input_ids': tokenized['input_ids'],
        'labels': tokenized['input_ids'].copy()
    }
    
    return result

def preprocess_evaluation_function_with_differential(
    examples, 
    tokenizer, 
    include_differential: bool = True,
    medical_knowledge: MedicalKnowledgeEnhancer = None
):
    """Preprocess evaluation data with differential diagnosis"""
    
    texts = []
    
    for i in range(len(examples['clinical_note'])):
        clinical_note = examples['clinical_note'][i]
        reasoning_chain = examples['reasoning_chain'][i]
        disease = examples['disease'][i]
        
        clinical_note = clinical_note[:1500]
        
        if include_differential and medical_knowledge:
            symptoms = medical_knowledge.extract_symptoms_from_note(clinical_note)
            differentials = medical_knowledge.generate_differential_diagnoses(disease, symptoms)
            
            instruction = """Analyze this clinical note and provide:
1. Most suspected disease (primary diagnosis)
2. Differential diagnoses (other possible conditions to consider)
3. Diagnostic reasoning
4. Precautions for the primary diagnosis

Format your response as:
**Primary Diagnosis:** [disease name]
**Differential Diagnoses:** [list of other possible conditions]
**Reasoning:** [your reasoning chain]
**Precautions:** [precautions for primary diagnosis]"""
            
            if differentials:
                differential_text = ", ".join(differentials[:3])
            else:
                differential_text = "Consider other conditions with similar symptoms"
            
            target_response = f"""**Primary Diagnosis:** {disease}
**Differential Diagnoses:** {differential_text}
**Reasoning:** {reasoning_chain}
**Precautions:** [Expected precautions would be here]"""
        
        else:
            instruction = """Analyze this clinical note and provide:
1. Most suspected disease
2. Diagnostic reasoning
3. Precautions for the most suspected disease

Format your response as:
**Disease:** [disease name]
**Reasoning:** [your reasoning chain]
**Precautions:** [precautions]"""
            
            target_response = f"""**Disease:** {disease}
**Reasoning:** {reasoning_chain}
**Precautions:** [Expected precautions would be here]"""
        
        full_text = f"<|im_start|>user\n{instruction}\n\nClinical Note:\n{clinical_note}<|im_end|>\n<|im_start|>assistant\n{target_response}<|im_end|>"
        texts.append(full_text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding=False,
        max_length=512,
        return_tensors=None,
        add_special_tokens=True,
    )
    
    result = {
        'input_ids': tokenized['input_ids'],
        'labels': tokenized['input_ids'].copy()
    }
    
    return result

class CustomDataCollator:
    """Custom data collator"""
    
    def __init__(self, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __call__(self, features):
        input_ids = [feature['input_ids'] for feature in features]
        labels = [feature['labels'] for feature in features]
        
        batch_max_length = max(len(seq) for seq in input_ids)
        batch_max_length = min(batch_max_length, self.max_length)
        
        padded_input_ids = []
        padded_attention_mask = []
        padded_labels = []
        
        for i in range(len(input_ids)):
            input_seq = input_ids[i]
            label_seq = labels[i]
            
            if len(input_seq) > batch_max_length:
                input_seq = input_seq[:batch_max_length]
                label_seq = label_seq[:batch_max_length]
            
            pad_length = batch_max_length - len(input_seq)
            
            padded_input = input_seq + [self.tokenizer.pad_token_id] * pad_length
            padded_attention = [1] * len(input_seq) + [0] * pad_length
            padded_label = label_seq + [-100] * pad_length
            
            padded_input_ids.append(padded_input)
            padded_attention_mask.append(padded_attention)
            padded_labels.append(padded_label)
        
        batch = {
            'input_ids': torch.tensor(padded_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(padded_attention_mask, dtype=torch.long),
            'labels': torch.tensor(padded_labels, dtype=torch.long),
        }
        
        return batch

def compute_metrics_with_differential(eval_pred, precautions_evaluator=None, medical_knowledge=None):
    """Compute evaluation metrics including differential diagnosis evaluation"""
    predictions, labels = eval_pred
    
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    metrics = {
        "eval_samples": len(predictions)
    }
    
    # Evaluate differential diagnosis if medical knowledge is available
    if medical_knowledge:
        differential_scores = []
        has_differential = []
        
        for pred_text, label_text in zip(decoded_preds[:10], decoded_labels[:10]):
            # Extract true disease from label
            disease_match = re.search(r"\*\*(?:Primary Diagnosis|Disease):\*\*\s*(.+)", label_text)
            if disease_match:
                true_disease = disease_match.group(1).strip()
                
                # Extract predicted differentials
                diff_match = re.search(r"\*\*Differential Diagnoses:\*\*\s*(.+)", pred_text)
                if diff_match:
                    predicted_differentials = [d.strip() for d in diff_match.group(1).split(',')]
                    
                    # Generate expected differentials
                    symptoms = []  # Would need clinical note here
                    expected_differentials = medical_knowledge.generate_differential_diagnoses(
                        true_disease, symptoms
                    )
                    
                    # Calculate overlap
                    if expected_differentials and predicted_differentials:
                        overlap = len(set(p.lower() for p in predicted_differentials) 
                                   & set(e.lower() for e in expected_differentials))
                        score = overlap / len(expected_differentials) if expected_differentials else 0
                        differential_scores.append(score)
                        has_differential.append(1)
                    else:
                        has_differential.append(0)
        
        if differential_scores:
            metrics.update({
                "avg_differential_score": np.mean(differential_scores),
                "has_differential_rate": np.mean(has_differential),
            })
    
    # Evaluate precautions if available
    if precautions_evaluator:
        precautions_scores = []
        has_precautions = []
        
        for pred_text, label_text in zip(decoded_preds[:10], decoded_labels[:10]):
            disease_match = re.search(r"\*\*(?:Primary Diagnosis|Disease):\*\*\s*(.+)", label_text)
            if disease_match:
                true_disease = disease_match.group(1).strip()
                eval_result = precautions_evaluator.evaluate_precautions_match(pred_text, true_disease)
                precautions_scores.append(eval_result.get("match_score", 0.0))
                has_precautions.append(eval_result.get("has_precautions", 0.0))
        
        if precautions_scores:
            metrics.update({
                "avg_precautions_match": np.mean(precautions_scores),
                "has_precautions_rate": np.mean(has_precautions),
            })
    
    return metrics

def main():
    # Configuration
    TRAINING_CSV = "direct_dataset_simplified.csv"
    PRECAUTIONS_CSV = "disease_precaution.csv"
    SYMPTOMS_CSV = "symptoms_to_diseases.csv"  # Optional: symptoms mapping
    OUTPUT_DIR = "./qwen1.5b-clinical-reasoning-differential"
    PHASE1_MODEL_PATH = "./qwen1.5b-symptoms-precautions"
    
    # Set this to True to include differential diagnosis
    INCLUDE_DIFFERENTIAL_DIAGNOSIS = True
    
    # Clear memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    print("="*80)
    print("PHASE 2: CLINICAL REASONING WITH DIFFERENTIAL DIAGNOSIS")
    print("="*80)
    
    # Load medical knowledge for differential diagnosis
    medical_knowledge = None
    if INCLUDE_DIFFERENTIAL_DIAGNOSIS:
        print("\nInitializing medical knowledge for differential diagnosis...")
        medical_knowledge = MedicalKnowledgeEnhancer(SYMPTOMS_CSV if os.path.exists(SYMPTOMS_CSV) else None)
        print("Medical knowledge loaded")
    
    # Load data
    df, precautions_evaluator = load_training_data(TRAINING_CSV, PRECAUTIONS_CSV)
    
    if len(df) == 0:
        print("No training data found!")
        return
    
    # Split data
    train_df, eval_df = train_test_split(
        df, 
        test_size=0.2, 
        random_state=42, 
        stratify=df['disease'] if 'disease' in df.columns else None
    )
    
    print(f"\nTraining set: {len(train_df)} examples")
    print(f"Validation set: {len(eval_df)} examples")
    
    # Create datasets
    train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
    eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))
    
    dataset_dict = DatasetDict({
        "train": train_dataset,
        "validation": eval_dataset
    })
    
    # Load model with frozen phase 1 adapters
    print("\nLoading model with frozen phase 1 adapters...")
    model, tokenizer = load_model_with_frozen_adapters(PHASE1_MODEL_PATH)
    
    # Preprocess functions
    def preprocess_train_batch(examples):
        return preprocess_training_function_with_differential(
            examples, tokenizer, medical_knowledge, INCLUDE_DIFFERENTIAL_DIAGNOSIS
        )
    
    def preprocess_eval_batch(examples):
        return preprocess_evaluation_function_with_differential(
            examples, tokenizer, INCLUDE_DIFFERENTIAL_DIAGNOSIS, medical_knowledge
        )
    
    print(f"\nTokenizing data (Differential Diagnosis: {'ENABLED' if INCLUDE_DIFFERENTIAL_DIAGNOSIS else 'DISABLED'})...")
    tokenized_train = dataset_dict["train"].map(
        preprocess_train_batch,
        batched=True,
        remove_columns=dataset_dict["train"].column_names,
        desc="Tokenizing training data"
    )
    
    tokenized_eval = dataset_dict["validation"].map(
        preprocess_eval_batch,
        batched=True,
        remove_columns=dataset_dict["validation"].column_names,
        desc="Tokenizing evaluation data"
    )
    
    print(f"\nTokenized training examples: {len(tokenized_train)}")
    print(f"Tokenized validation examples: {len(tokenized_eval)}")
    
    # Data collator
    data_collator = CustomDataCollator(tokenizer, max_length=512)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        num_train_epochs=3,
        logging_dir="./logs",
        logging_steps=10,
        eval_steps=50,
        save_steps=100,
        eval_strategy="steps",
        save_strategy="steps",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        warmup_steps=20,
        fp16=True,
        dataloader_pin_memory=False,
        save_total_limit=1,
        remove_unused_columns=False,
        report_to="none",
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        dataloader_drop_last=True,
        gradient_checkpointing=True,
        eval_accumulation_steps=2,
    )
    
    # Create compute_metrics function
    def compute_metrics_wrapper(eval_pred):
        return compute_metrics_with_differential(
            eval_pred, 
            precautions_evaluator, 
            medical_knowledge
        )
    
    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_wrapper if (precautions_evaluator or medical_knowledge) else None,
    )
    
    # Train
    print("\n" + "="*80)
    print("STARTING TRAINING")
    print("="*80)
    print(f"Differential Diagnosis: {'ENABLED' if INCLUDE_DIFFERENTIAL_DIAGNOSIS else 'DISABLED'}")
    print(f"Precautions Evaluation: {'ENABLED' if precautions_evaluator else 'DISABLED'}")
    
    try:
        train_result = trainer.train()
        
        # Save model
        print("\nSaving model...")
        trainer.save_model()
        tokenizer.save_pretrained(OUTPUT_DIR)
        model.save_pretrained(os.path.join(OUTPUT_DIR, "final_adapters"))
        
        print(f"\n{'='*80}")
        print("TRAINING COMPLETED!")
        print(f"{'='*80}")
        print(f"Model saved in: {OUTPUT_DIR}")
        
        # Generate sample predictions to show differential diagnosis
        print("\nGENERATING SAMPLE PREDICTIONS WITH DIFFERENTIAL DIAGNOSIS:")
        print("-"*60)
        
        sample_data = eval_df.head(3)
        for idx, row in sample_data.iterrows():
            print(f"\nSample {idx+1}:")
            print(f"True Disease: {row['disease']}")
            print(f"Clinical Note (first 100 chars): {row['clinical_note'][:100]}...")
            
            # Generate prediction with differential diagnosis prompt
            if INCLUDE_DIFFERENTIAL_DIAGNOSIS:
                prompt = f"""<|im_start|>user
Analyze this clinical note and provide:
1. Most suspected disease (primary diagnosis)
2. Differential diagnoses (other possible conditions to consider)
3. Diagnostic reasoning
4. Precautions for the primary diagnosis

Format your response as:
**Primary Diagnosis:** [disease name]
**Differential Diagnoses:** [list of other possible conditions]
**Reasoning:** [your reasoning chain]
**Precautions:** [precautions for primary diagnosis]

Clinical Note:
{row['clinical_note'][:500]}<|im_end|>
<|im_start|>assistant
"""
            else:
                prompt = f"""<|im_start|>user
Analyze this clinical note and provide:
1. Most suspected disease
2. Diagnostic reasoning
3. Precautions for the most suspected disease

Format your response as:
**Disease:** [disease name]
**Reasoning:** [your reasoning chain]
**Precautions:** [precautions]

Clinical Note:
{row['clinical_note'][:500]}<|im_end|>
<|im_start|>assistant
"""
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=300,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9
                )
            
            prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = prediction.split("assistant")[-1].strip()
            print(f"\nGenerated Response:")
            print(response)
            
            # Evaluate if we have precautions evaluator
            if precautions_evaluator:
                eval_result = precautions_evaluator.evaluate_precautions_match(response, row['disease'])
                print(f"\nPrecautions Match Score: {eval_result.get('match_score', 0):.3f}")
            
            print("-"*60)
        
    except Exception as e:
        print(f"Training failed with error: {e}")
        import traceback
        traceback.print_exc()

def test_differential_diagnosis_generation():
    """Test the differential diagnosis generation separately"""
    print("\n" + "="*80)
    print("TESTING DIFFERENTIAL DIAGNOSIS GENERATION")
    print("="*80)
    
    # Create medical knowledge enhancer
    medical_knowledge = MedicalKnowledgeEnhancer()
    
    # Test cases
    test_cases = [
        {
            "clinical_note": "Patient presents with fever, cough, and shortness of breath for 3 days. Chest X-ray shows infiltrates.",
            "true_disease": "pneumonia"
        },
        {
            "clinical_note": "55-year-old male with chest pain radiating to left arm, diaphoresis, and nausea.",
            "true_disease": "myocardial infarction"
        },
        {
            "clinical_note": "Patient with unilateral headache, photophobia, and nausea. History of migraines.",
            "true_disease": "migraine"
        }
    ]
    
    for i, test_case in enumerate(test_cases):
        print(f"\nTest Case {i+1}:")
        print(f"Clinical Note: {test_case['clinical_note'][:100]}...")
        print(f"True Disease: {test_case['true_disease']}")
        
        # Extract symptoms
        symptoms = medical_knowledge.extract_symptoms_from_note(test_case['clinical_note'])
        print(f"Extracted Symptoms: {symptoms}")
        
        # Generate differentials
        differentials = medical_knowledge.generate_differential_diagnoses(
            test_case['true_disease'], symptoms
        )
        print(f"Generated Differential Diagnoses: {differentials}")
        
        print("-"*60)

if __name__ == "__main__":
    # Uncomment to test differential diagnosis generation
    # test_differential_diagnosis_generation()
    
    # Run training
    main()